# Banking77 Fine-Tuning: QLoRA on Qwen2.5-1.5B-Instruct

This notebook fine-tunes Qwen/Qwen2.5-1.5B-Instruct on the Banking77 intent classification dataset using QLoRA.

**Environment:** Google Colab with T4 GPU (free tier)

## Steps:
1. Install dependencies
2. Download and prepare Banking77 dataset
3. Configure QLoRA and load model
4. Train with SFTTrainer
5. Evaluate on validation set
6. Log to MLflow
7. Merge LoRA adapters and export
8. Package artifacts for download

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    datasets accelerate bitsandbytes mlflow scikit-learn \
    torch --upgrade

In [ ]:
# Step 2: Configuration
import os
import json
from datetime import datetime, timezone

CONFIG = {
    'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
    'dataset': 'PolyAI/banking77',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'learning_rate': 2e-4,
    'epochs': 2,
    'max_seq_length': 512,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'seed': 42,
    'output_dir': './training_output',
    'merged_dir': './merged_model',
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# Step 3: Load and prepare dataset
from datasets import load_dataset
import random

dataset = load_dataset('PolyAI/banking77')
label_names = dataset['train'].features['label'].names
print(f'Train: {len(dataset["train"])}, Test: {len(dataset["test"])}')
print(f'Classes: {len(label_names)}')

# Create validation split
random.seed(CONFIG['seed'])
train_ds = dataset['train'].shuffle(seed=CONFIG['seed'])
val_size = 500
val_ds = train_ds.select(range(val_size))
train_ds = train_ds.select(range(val_size, len(train_ds)))
print(f'Train: {len(train_ds)}, Validation: {len(val_ds)}')

In [ ]:
# Step 4: Format as instruction dataset
SYSTEM_PROMPT = ('You are a banking customer intent classifier. '
    'Given a customer message, respond with exactly one intent label. '
    'Output only the intent label, nothing else.')

def format_example(example):
    label = label_names[example['label']]
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['text']},
            {'role': 'assistant', 'content': label},
        ]
    }

train_formatted = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_formatted = val_ds.map(format_example, remove_columns=val_ds.column_names)
print(f'Formatted train: {len(train_formatted)}, val: {len(val_formatted)}')
print('Sample:', train_formatted[0])

In [ ]:
# Step 5: Load model with QLoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_rank'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    max_seq_length=CONFIG['max_seq_length'],
    seed=CONFIG['seed'],
    fp16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

import time
start_time = time.time()
train_result = trainer.train()
training_duration = time.time() - start_time
print(f'Training completed in {training_duration:.0f}s')
print(f'Final train loss: {train_result.training_loss:.4f}')

In [ ]:
# Step 7: Evaluate on validation set
from sklearn.metrics import accuracy_score, f1_score

def evaluate_model(model, tokenizer, dataset, label_names, max_samples=500):
    model.eval()
    predictions, ground_truth = [], []
    
    for i, example in enumerate(dataset):
        if i >= max_samples:
            break
        messages = example['messages'][:2]  # system + user only
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=CONFIG['max_seq_length']).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=30, temperature=0.0, do_sample=False)
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        gt = example['messages'][2]['content']
        predictions.append(response)
        ground_truth.append(gt)
    
    accuracy = accuracy_score(ground_truth, predictions)
    macro_f1 = f1_score(ground_truth, predictions, average='macro', zero_division=0)
    invalid = sum(1 for p in predictions if p not in label_names) / len(predictions)
    
    return {'accuracy': accuracy, 'macro_f1': macro_f1, 'invalid_rate': invalid, 'total': len(predictions)}

val_metrics = evaluate_model(model, tokenizer, val_formatted, label_names)
print(f'Validation Results:')
print(f'  Accuracy:     {val_metrics["accuracy"]:.4f}')
print(f'  Macro F1:     {val_metrics["macro_f1"]:.4f}')
print(f'  Invalid Rate: {val_metrics["invalid_rate"]:.4f}')

In [ ]:
# Step 8: Log to MLflow
import mlflow

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('banking77-finetuning')

with mlflow.start_run(run_name=f'qwen2.5-1.5b-qlora-{datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")}'):
    mlflow.log_params(CONFIG)
    mlflow.log_params({'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})
    mlflow.log_metric('train_loss', train_result.training_loss)
    mlflow.log_metric('training_duration_s', training_duration)
    mlflow.log_metrics(val_metrics)
    print('Logged to MLflow')

In [ ]:
# Step 9: Merge LoRA adapters and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained(CONFIG['merged_dir'])
tokenizer.save_pretrained(CONFIG['merged_dir'])
print(f'Merged model saved to {CONFIG["merged_dir"]}')

In [ ]:
# Step 10: Package for download
!tar -czf merged_model.tar.gz -C {CONFIG['merged_dir']} .

# Copy MLflow artifacts
!tar -czf mlflow_artifacts.tar.gz mlruns/

print('\n📦 Download these files:')
print('1. merged_model.tar.gz — Merged model for GGUF conversion')
print('2. mlflow_artifacts.tar.gz — MLflow experiment logs')
print('\nNext steps:')
print('1. Download both files')
print('2. Extract merged_model.tar.gz to your local project')
print('3. Run: scripts/convert_to_gguf.sh')
print('4. Run: scripts/quantize_model.sh')
print('5. Start llama.cpp server with the quantized model')